In [1]:
!pip install xgboost lightgbm catboost pandas numpy scikit-learn

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (accuracy_score, classification_report,
                             mean_squared_error, r2_score)
from sklearn.model_selection import StratifiedKFold, KFold, cross_validate


In [3]:
# load data and importance scores
df_final = pd.read_csv('steam_finalized_dataset.csv')
importance_df = pd.read_csv('feature_importances.csv')

In [4]:
import numpy as np

def get_top_correlations(df, top_n=30, threshold=0.90, verbose=False):
    corr_matrix = df.corr().abs()  # Get absolute correlation values
    corr_pairs = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))  # upper triangle only
    .stack()
    .reset_index()
    )   
    corr_pairs.columns = ['Feature 1', 'Feature 2', 'Correlation']
    corr_pairs = corr_pairs.sort_values('Correlation', ascending=False)

    if verbose:
        print("Top 30 most correlated feature pairs:")
        print(corr_pairs.head(30).to_string(index=False))
    
    to_drop = get_highly_correlated_features(corr_pairs, threshold=threshold, verbose=verbose)

    return to_drop

    
def get_highly_correlated_features(corr_pairs, threshold=0.90, verbose=False):
    to_drop = set()

    for _, row in corr_pairs.iterrows():
        f1, f2, corr = row['Feature 1'], row['Feature 2'], row['Correlation']
        if corr < threshold:
            break  # already sorted descending, so we can stop early
        # Drop f2 (keep f1) — unless f1 is already marked for dropping
        if f1 not in to_drop:
            to_drop.add(f2)

    if verbose:
        print(f"Features to drop ({len(to_drop)}):")
        print(sorted(to_drop))
    
    return to_drop


In [5]:
# feature selection
threshold = 0.01
selected_features = importance_df[
    importance_df['importance'] > threshold
]['feature'].tolist()
# drop_features = {'developer', 'publisher'}

In [6]:
# targets 
le = LabelEncoder()
threshold = 0.90
verbose = True
n = 30 

to_drop = get_top_correlations(df_final[selected_features], top_n=n, threshold=threshold, verbose=verbose)

X = df_final[selected_features].drop(columns=to_drop)
y_reg = df_final['wilson_score']
y_clf = le.fit_transform(df_final['rating_category'])

# stratified test/train split (stratify by classification target i.e rating category)
X_train, X_test, y_clf_train, y_clf_test, y_reg_train, y_reg_test = train_test_split(
    X, y_clf, y_reg,
    test_size=0.2,
    random_state=42,
    stratify=y_clf       
)

# Verify stratification worked!!
print("Train distribution:")
print(pd.Series(le.inverse_transform(y_clf_train)).value_counts(normalize=True).round(3))
print("\nTest distribution:")
print(pd.Series(le.inverse_transform(y_clf_test)).value_counts(normalize=True).round(3))

Top 30 most correlated feature pairs:
    Feature 1     Feature 2  Correlation
    developer     publisher     0.898699
support_email     publisher     0.839683
 singleplayer     adventure     0.808875
    developer support_email     0.797625
    developer  release_year     0.686637
    adventure        action     0.686462
 singleplayer        action     0.657727
    publisher  release_year     0.623158
support_email  release_year     0.581157
        indie  singleplayer     0.560417
      website   support_url     0.560363
   owners_log  release_year     0.513858
        indie     adventure     0.491484
   owners_log     developer     0.461409
   owners_log     publisher     0.432441
   owners_log support_email     0.402955
        indie        casual     0.365794
    developer   support_url     0.364935
    publisher   support_url     0.358808
  support_url  release_year     0.352764
        price    storage_mb     0.349902
 singleplayer        casual     0.338321
    adventure      

In [7]:
# Raw versions 
X_train_raw = X_train.copy()
X_test_raw  = X_test.copy()

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

class TargetEncoderCV(BaseEstimator, TransformerMixin):
    """Target encodes a column using only the fold's training data."""
    def __init__(self, col):
        self.col = col
        
    def fit(self, X, y):
        X_ = X.copy() if hasattr(X, 'copy') else pd.DataFrame(X)
        self.enc_map_    = pd.Series(y).groupby(
            X_[self.col] if isinstance(X_, pd.DataFrame) else X_[:, self.col]
        ).mean()
        self.global_mean_ = float(pd.Series(y).mean())
        return self
    
    def transform(self, X):
        X_ = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        X_['developer_enc'] = X_[self.col].map(self.enc_map_).fillna(self.global_mean_)
        X_.drop(columns=[self.col], inplace=True)
        return X_


def make_clf_pipeline(model, scale=False):
    steps = [('target_enc', TargetEncoderCV(col='developer'))]
    # steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)

def make_reg_pipeline(model, scale=False):
    steps = [('target_enc', TargetEncoderCV(col='developer'))]
    # steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)

In [9]:
# # Scaling
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled  = scaler.transform(X_test)

# Cross validation setup
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

In [10]:
clf_models = {
    'Naive Bayes':         GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=200, random_state=42,
                                         eval_metric='mlogloss', verbosity=0),
    'KNN':                 KNeighborsClassifier(n_neighbors=10, n_jobs=-1),
}

reg_models = {
    'Linear Regression':   LinearRegression(),
    'Random Forest':       RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':             XGBRegressor(n_estimators=200, random_state=42, verbosity=0),
}


clf_pipelines = {
    'Naive Bayes':         make_clf_pipeline(GaussianNB(), scale=True),
    'Logistic Regression': make_clf_pipeline(LogisticRegression(max_iter=1000, random_state=42), scale=True),
    'Random Forest':       make_clf_pipeline(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    'XGBoost':             make_clf_pipeline(XGBClassifier(n_estimators=200, random_state=42,
                                         eval_metric='mlogloss', verbosity=0)),
    'KNN':                 make_clf_pipeline(KNeighborsClassifier(n_neighbors=10, n_jobs=-1), scale=True),
}

reg_pipelines = {
    'Linear Regression': make_reg_pipeline(LinearRegression(), scale=True),
    'Random Forest':     make_reg_pipeline(RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
     'XGBoost':             make_reg_pipeline(XGBRegressor(n_estimators=200, random_state=42, verbosity=0)),
}

In [11]:
# CLASSIFICATION — 5-Fold CV
print("CLASSIFICATION — 5-Fold Stratified CV")
print("=" * 50)

clf_cv_results = {}
for name, pipe in clf_pipelines.items():
    scores = cross_validate(
        pipe, X_train_raw, y_clf_train,
        cv=cv_clf,
        scoring=['accuracy', 'f1_weighted'],
        n_jobs=-1
    )
    clf_cv_results[name] = {
        'CV Acc (mean)': scores['test_accuracy'].mean(),
        'CV Acc (std)':  scores['test_accuracy'].std(),
        'CV F1 (mean)':  scores['test_f1_weighted'].mean(),
        'CV F1 (std)':   scores['test_f1_weighted'].std(),
    }
    print(f"{name:22s} | Acc = {scores['test_accuracy'].mean():.4f} "
          f"± {scores['test_accuracy'].std():.4f} | "
          f"F1 = {scores['test_f1_weighted'].mean():.4f} "
          f"± {scores['test_f1_weighted'].std():.4f}")

CLASSIFICATION — 5-Fold Stratified CV
Naive Bayes            | Acc = 0.2184 ± 0.0023 | F1 = 0.1918 ± 0.0047
Logistic Regression    | Acc = 0.3565 ± 0.0088 | F1 = 0.3176 ± 0.0070
Random Forest          | Acc = 0.3969 ± 0.0046 | F1 = 0.3893 ± 0.0043
XGBoost                | Acc = 0.3823 ± 0.0083 | F1 = 0.3795 ± 0.0083
KNN                    | Acc = 0.3281 ± 0.0053 | F1 = 0.3186 ± 0.0050


In [12]:
# REGRESSION — 5-Fold CV
print("REGRESSION — 5-Fold KFold CV")
print("=" * 50)

reg_cv_results = {}
for name, pipe in reg_pipelines.items():
    scores = cross_validate(
        pipe, X_train_raw, y_reg_train,
        cv=cv_reg,
        scoring=['r2', 'neg_root_mean_squared_error'],
        n_jobs=-1
    )
    reg_cv_results[name] = {
        'CV R² (mean)':   scores['test_r2'].mean(),
        'CV R² (std)':    scores['test_r2'].std(),
        'CV RMSE (mean)': -scores['test_neg_root_mean_squared_error'].mean(),
        'CV RMSE (std)':  scores['test_neg_root_mean_squared_error'].std(),
    }
    print(f"{name:22s} | R² = {scores['test_r2'].mean():.4f} "
          f"± {scores['test_r2'].std():.4f} | "
          f"RMSE = {-scores['test_neg_root_mean_squared_error'].mean():.4f} "
          f"± {scores['test_neg_root_mean_squared_error'].std():.4f}")

REGRESSION — 5-Fold KFold CV
Linear Regression      | R² = 0.1778 ± 0.0043 | RMSE = 0.2300 ± 0.0012
Random Forest          | R² = 0.2177 ± 0.0061 | RMSE = 0.2243 ± 0.0010
XGBoost                | R² = 0.2320 ± 0.0046 | RMSE = 0.2223 ± 0.0008


In [13]:
print("\nFINAL TEST SET — CLASSIFICATION")
print("=" * 50)

clf_test_results = {}
for name, pipe in clf_pipelines.items():
    pipe.fit(X_train_raw, y_clf_train)
    preds = pipe.predict(X_test_raw)
    acc = accuracy_score(y_clf_test, preds)
    clf_test_results[name] = acc
    print(f"\n{name} — Test Accuracy: {acc:.4f}")
    print(classification_report(y_clf_test, preds, target_names=le.classes_))

print("\nFINAL TEST SET — REGRESSION")
print("=" * 50)

reg_test_results = {}
for name, pipe in reg_pipelines.items():
    pipe.fit(X_train_raw, y_reg_train)
    preds = pipe.predict(X_test_raw)
    r2   = r2_score(y_reg_test, preds)
    rmse = np.sqrt(mean_squared_error(y_reg_test, preds))
    reg_test_results[name] = {'R²': r2, 'RMSE': rmse}
    print(f"{name:22s} | R² = {r2:.4f} | RMSE = {rmse:.4f}")


FINAL TEST SET — CLASSIFICATION

Naive Bayes — Test Accuracy: 0.2164
                 precision    recall  f1-score   support

          Mixed       0.23      0.11      0.15      1583
Mostly Negative       0.19      0.22      0.21      1162
Mostly Positive       0.35      0.07      0.11      1115
       Negative       0.18      0.85      0.30       609
       Positive       0.63      0.16      0.25       896

       accuracy                           0.22      5365
      macro avg       0.32      0.28      0.20      5365
   weighted avg       0.31      0.22      0.19      5365


Logistic Regression — Test Accuracy: 0.3620
                 precision    recall  f1-score   support

          Mixed       0.34      0.65      0.45      1583
Mostly Negative       0.34      0.35      0.35      1162
Mostly Positive       0.33      0.13      0.18      1115
       Negative       0.29      0.01      0.01       609
       Positive       0.52      0.40      0.45       896

       accuracy          

In [14]:
# Summary Tables
print("\n\nCLASSIFICATION SUMMARY")
clf_summary = pd.DataFrame(clf_cv_results).T
clf_summary['Test Accuracy'] = pd.Series(clf_test_results)
print(clf_summary.sort_values('CV Acc (mean)', ascending=False).round(4))

print("\nREGRESSION SUMMARY")
reg_summary = pd.DataFrame(reg_cv_results).T
reg_summary['Test R²']   = pd.Series({k: v['R²']   for k, v in reg_test_results.items()})
reg_summary['Test RMSE'] = pd.Series({k: v['RMSE'] for k, v in reg_test_results.items()})
print(reg_summary.sort_values('CV R² (mean)', ascending=False).round(4))



CLASSIFICATION SUMMARY
                     CV Acc (mean)  CV Acc (std)  CV F1 (mean)  CV F1 (std)  \
Random Forest               0.3969        0.0046        0.3893       0.0043   
XGBoost                     0.3823        0.0083        0.3795       0.0083   
Logistic Regression         0.3565        0.0088        0.3176       0.0070   
KNN                         0.3281        0.0053        0.3186       0.0050   
Naive Bayes                 0.2184        0.0023        0.1918       0.0047   

                     Test Accuracy  
Random Forest               0.3976  
XGBoost                     0.3905  
Logistic Regression         0.3620  
KNN                         0.3323  
Naive Bayes                 0.2164  

REGRESSION SUMMARY
                   CV R² (mean)  CV R² (std)  CV RMSE (mean)  CV RMSE (std)  \
XGBoost                  0.2320       0.0046          0.2223         0.0008   
Random Forest            0.2177       0.0061          0.2243         0.0010   
Linear Regression    